In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
prompts = [
    {"role": "system", "content" : "You are a Langchain expert."},
    {"role":"user", "content" : "What are the important topics to learn in Langchain ?"}
]
res = llm.invoke(prompts)
print(res.content)

Prompt Templates - Dynamic prompts

In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
load_dotenv()
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
prompt_template = ChatPromptTemplate.from_messages(
    [
        {"role":"system", "content" : "You are a {coding_language} expert."},
        {"role":"user", "content" : "{question}"}
    ]
)
print(prompt_template)

input_variables=['coding_language', 'question'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['coding_language'], input_types={}, partial_variables={}, template='You are a {coding_language} expert.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]


In [ ]:
final_prompt = prompt_template.format_messages(coding_language = "cpp", question ="what is the key difference b/w vector and set ?")
res = llm.invoke(final_prompt)
print(res.content)

In [ ]:
final_prompt = prompt_template.format_messages(coding_language = "language translation", question ="Translate the below sentence to Spanish : I am learning Langchain.")
res = llm.invoke(final_prompt)
print(res.content)

Chains - runnable

In [ ]:
prompt_template.invoke({"coding_language" : "language translation", "question" : "Translate the below sentence to Spanish : I am learning Langchain."})
# prompt_template
res = llm.invoke(prompt_template.invoke({"coding_language" : "language translation", "question" : "Translate the below sentence to Hinglish : I am learning Langchain."}))
print(res.content)

In [ ]:
chains = prompt_template | llm    ##In above code you can see prompt_template and llm are separate and we are invoking them separately but with chains we can combine them and invoke together.
res = chains.invoke({"coding_language" : "language translation", "question" : "Translate the below sentence to Hinglish : I am learning Langchain."})
print(res.content)

In [ ]:
def transform_case(text : str):
    return text.upper()

out = StrOutputParser()
chains = prompt_template | llm | out | transform_case   
res = chains.invoke({"coding_language" : "language translation", "question" : "Translate the below sentence to Hinglish : I am learning Langchain."})
print(res)

Langchain advanced pipelines - Runnable passthrough and Runnable Parallel

In [25]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

# 🔹 Parser
parser = StrOutputParser()

# =========================
# 🔥 PROMPTS (Chat Style)
# =========================

sentiment_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a financial analyst."),
    ("user", "Analyze sentiment of this news:\n\n{news}\n\nReply: Positive/Negative/Neutral")
])

summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a stock expert."),
    ("user", "Summarize this news in 2 lines:\n\n{news}")
])

prediction_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a stock advisor."),
    ("user", """Predict stock movement:

{news}

Output:
- Prediction (Up/Down)
- Reason
""")
])

# =========================
# 🔥 YOUR STRUCTURE
# =========================

stock_chain = (
    RunnablePassthrough()
    .assign(
        analysis = RunnableParallel(
            sentiment = sentiment_prompt | llm | parser,
            summary = summary_prompt | llm | parser,
        ),
        prediction = prediction_prompt | llm | parser
    )
)

# =========================
# 🔹 INPUT
# =========================

input_data = {
    "news": "Infosys announced a major AI partnership and expects strong revenue growth next year."
}

# =========================
# 🔹 RUN
# =========================

result = stock_chain.invoke(input_data)

print("\n🔹 FINAL OUTPUT:\n")
print(result)


🔹 FINAL OUTPUT:

{'news': 'Infosys announced a major AI partnership and expects strong revenue growth next year.', 'analysis': {'sentiment': 'Positive', 'summary': 'Infosys has forged a significant AI partnership, anticipating robust revenue growth in the upcoming year. This strategic move positions the company for expansion and innovation in the artificial intelligence sector.'}, 'prediction': '- **Prediction:** Up\n\n- **Reason:** The announcement of a major AI partnership is a significant positive development for Infosys. AI is a high-growth sector, and a strong partnership in this area suggests potential for new revenue streams and increased demand for their services. Coupled with expectations of strong revenue growth next year, this indicates positive future performance, which typically leads to an upward movement in stock price. Investors often react favorably to news that signals innovation and future profitability.'}
